In [213]:
%matplotlib TkAgg
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.special import hankel2
from IPython.display import HTML

In [ ]:
# Wave speed
# Fundamentele constanten
eps0 = 8.854187817e-12   # F/m
mu0  = 4*np.pi*1e-7      # H/m
c   = 1/np.sqrt(eps0*mu0)

# Domain parameters
domain_width = 3*10**7
domain_height = 3*10**7

# Frequency range
sigma = 0.03*10**-1
f_max = (5/sigma)/(2*np.pi)

# Grid
lambda_min = c / f_max       # minimum wavelength
dx = lambda_min / 50         # spatial step
dy = dx
CFL = 0.999999
dt = CFL / (c*np.sqrt(1/dx**2 + 1/dy**2))     # CFL condition

Nx, Ny = round(domain_width / dx), round(domain_height / dy)
Nt = round((35 * sigma / dt))    # max time steps

# Source
ps_x = int(domain_width/2/dx)
ps_y = int(domain_height/2/dy)

J0 = 10
tc = 5*sigma
def ps(t):
    return J0*np.exp(-(t-tc)**2/(2*sigma**2))

# dielectric
diel_x = domain_width*5/2 #(middle of the object)
diel_y = domain_height/2 #(middle of the object)
diel_height = domain_height/2
diel_width = domain_width/5

# Observation points
po_x = int(domain_width/3/dx)
po_y = int(domain_height/3/dy)


print(f"Grid size: {Nx} x {Ny}, Time steps: {Nt}")
print(f"dx = {dx:.4f} m, dt = {dt:.6f} s")
print(f"Lambda_min = {lambda_min:.4f} m, points per wavelength = {lambda_min/dx:.1f}")

Grid size: 2124 x 2124, Time steps: 3151
dx = 14127.3868 m, dt = 0.000033 s
Lambda_min = 1130190.9404 m, points per wavelength = 80.0


In [215]:
# Definition of kappa
N_PML = int(Nx/4)
m = 3
eta0 = np.sqrt(mu0 / eps0)

# PML-conductiviteitsprofielen (gesplitst per richting)
sigma_max = (m + 1) / (2 * eta0 * N_PML * dx)*10
print(sigma_max)
sigma_m_max = sigma_max * mu0 / eps0

sigma_pml   = sigma_max   * (np.arange(N_PML + 1) / N_PML) ** m
sigma_m_pml = sigma_m_max * (np.arange(N_PML + 1) / N_PML) ** m

def sigma_PML(E_or_H):
    kappa_m = np.zeros(((Nx+1)-1*(E_or_H=='Hy'), (Ny+1)-1*(E_or_H=='Hx')))

    if E_or_H=='Ezx':
        kappa_m[:N_PML+1, :] = sigma_pml[::-1, np.newaxis]     # ::-1 to invert the kappa array to match the def of kappa; np.newaxis to match dimensions
        kappa_m[-N_PML-1:, :] = sigma_pml[:, np.newaxis]
        
    if E_or_H=='Ezy':
        kappa_m[:,-N_PML-1:] = sigma_pml[np.newaxis, :]
        kappa_m[:, :N_PML+1] = sigma_pml[np.newaxis, ::-1]
    
    if E_or_H=='Hx':
        kappa_m[:N_PML+1, :] = sigma_m_pml[::-1, np.newaxis]     # ::-1 to invert the kappa array to match the def of kappa; np.newaxis to match dimensions
        kappa_m[-N_PML-1:, :] = sigma_m_pml[:, np.newaxis]
        
    if E_or_H=='Hy':
        kappa_m[:,-N_PML-1:] = sigma_m_pml[np.newaxis, :]
        kappa_m[:, :N_PML+1] = sigma_m_pml[np.newaxis, ::-1]
    
    return kappa_m


7.076900229200692e-09


In [216]:
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# components = ['Ezx', 'Ezy', 'Hx', 'Hy']
# titles = ['$\\sigma_x$ voor $E_{zx}$', '$\\sigma_y$ voor $E_{zy}$',
#           '$\\sigma^*_y$ voor $H_x$', '$\\sigma^*_x$ voor $H_y$']

# for ax, comp, title in zip(axes.flatten(), components, titles):
#     data = sigma_PML(comp)
    
#     vmax = data.max()
#     im = ax.imshow(data.T, origin='lower', cmap='Blues',
#                    vmin=0, vmax=vmax,
#                    extent=[0, data.shape[0]*dx*1e3, 0, data.shape[1]*dy*1e3])
    
#     cbar = plt.colorbar(im, ax=ax)
#     cbar.set_label('$\\sigma$ [S/m]' if 'E' in comp else '$\\sigma^*$ [Ω/m]')
#     cbar.ax.axhline(y=vmax, color='red', lw=1.5, label=f'max={vmax:.3f}')
    
#     # PML-grenzen aanduiden
#     for lim in [N_PML*dx*1e3, (data.shape[0]-N_PML)*dx*1e3]:
#         ax.axvline(lim, color='red', lw=1, ls='--', alpha=0.8)
#     for lim in [N_PML*dy*1e3, (data.shape[1]-N_PML)*dy*1e3]:
#         ax.axhline(lim, color='red', lw=1, ls='--', alpha=0.8)

#     # Annoteer de gradiëntrichting in elke PML-zone
#     cx = data.shape[0] * dx * 1e3 / 2   # midden van het domein
#     cy = data.shape[1] * dy * 1e3 / 2

#     arrow_kw = dict(fontsize=7, color='red', ha='center',
#                     arrowprops=dict(arrowstyle='->', color='red', lw=1.2))

#     if 'x' in comp.lower() or comp == 'Hy':
#         # gradiënt in x-richting
#         ax.annotate('0', xy=(N_PML*dx*1e3, cy),
#                     xytext=(N_PML*dx*1e3 + 1.5, cy), **arrow_kw)
#         ax.annotate('0', xy=((data.shape[0]-N_PML)*dx*1e3, cy),
#                     xytext=((data.shape[0]-N_PML)*dx*1e3 - 1.5, cy), **arrow_kw)
#         ax.text(0.5*N_PML*dx*1e3, cy, f'max\n{vmax:.2f}',
#                 color='red', fontsize=7, ha='center', va='center')
#         ax.text((data.shape[0]-0.5*N_PML)*dx*1e3, cy, f'max\n{vmax:.2f}',
#                 color='red', fontsize=7, ha='center', va='center')
#     else:
#         # gradiënt in y-richting
#         ax.annotate('0', xy=(cx, N_PML*dy*1e3),
#                     xytext=(cx, N_PML*dy*1e3 + 1.5), **arrow_kw)
#         ax.annotate('0', xy=(cx, (data.shape[1]-N_PML)*dy*1e3),
#                     xytext=(cx, (data.shape[1]-N_PML)*dy*1e3 - 1.5), **arrow_kw)
#         ax.text(cx, 0.5*N_PML*dy*1e3, f'max\n{vmax:.2f}',
#                 color='red', fontsize=7, ha='center', va='center')
#         ax.text(cx, (data.shape[1]-0.5*N_PML)*dy*1e3, f'max\n{vmax:.2f}',
#                 color='red', fontsize=7, ha='center', va='center')

#     ax.set_title(f'{title}\nmax = {vmax:.4f}', fontsize=11)
#     ax.set_xlabel('x [mm]')
#     ax.set_ylabel('y [mm]')

# plt.suptitle('PML conductiviteitsprofielen — rood gestippeld: PML-grens\n'
#              'Licht = laag (≈0), Donker = hoog (= max)', fontsize=12, y=1.02)
# plt.tight_layout()
# plt.show()

In [217]:
    # Materiaalmatrices (relatieve permittiviteit, permeabiliteit, geleiding)
    eps_r = np.ones((Nx+1, Ny+1))
    mu_r_Hx  = np.ones((Nx+1, Ny))
    mu_r_Hy  = np.ones((Nx, Ny+1))
    sigma_e = np.zeros((Nx, Ny))  # elektrische geleiding [S/m]

    # ── Voeg een rechthoekig diëlektrisch object toe ───────────────
    # Pas aan voor andere geometrieën
    USE_OBJECT = False

    if USE_OBJECT:
        obj_x0, obj_x1 = Nx//2 + 5,  Nx//2 + 35   # x-bereik (indices)
        obj_y0, obj_y1 = Ny//2 - 15, Ny//2 + 15   # y-bereik (indices)
        eps_r[obj_x0:obj_x1, obj_y0:obj_y1]   = 1   # relatieve permittiviteit
        sigma_e[obj_x0:obj_x1, obj_y0:obj_y1] = 0.0   # geleiding (0 = verliesvrij)

    print('Materiaalmatrices aangemaakt.')
    if USE_OBJECT:
        print(f'Object: eps_r={eps_r[obj_x0+1, obj_y0+1]:.1f}, '
            f'afmeting {(obj_x1-obj_x0)*dx*1e3:.0f}x{(obj_y1-obj_y0)*dy*1e3:.0f} mm')

Materiaalmatrices aangemaakt.


In [218]:
# ── Coëfficiënten voor H_x  (afhankelijk van sigma_y*, mu_r) ──
# sigma_y* is gedefinieerd op de y-as → maak 2D door broadcasting
smx  = sigma_PML('Hx')

denom_hx = 1 + smx*dt/(2*mu0*mu_r_Hx)
Chxa = (1 - smx*dt/(2*mu0*mu_r_Hx)) / denom_hx
Chxb = (dt/(mu0*mu_r_Hx)) / denom_hx / dy       # inclusief 1/dy

# ── Coëfficiënten voor H_y  (afhankelijk van sigma_x*, mu_r) ──
smy  = sigma_PML('Hy') # shape (Nx, Ny)

denom_hy = 1 + smy*dt/(2*mu0*mu_r_Hy)
Chya = (1 - smy*dt/(2*mu0*mu_r_Hy)) / denom_hy
Chyb = (dt/(mu0*mu_r_Hy)) / denom_hy / dx       # inclusief 1/dx

# ── Coëfficiënten voor E_zx  (afhankelijk van sigma_x, eps_r) ─
sx   = sigma_PML('Ezx') # shape (Nx, Ny)
epsr = eps_r                                   # shape (Nx, Ny)

denom_ezx = 1 + sx*dt/(2*eps0*epsr)
Cexa = (1 - sx*dt/(2*eps0*epsr)) / denom_ezx
Cexb = (dt/(eps0*epsr)) / denom_ezx / dx     # inclusief 1/dx

# ── Coëfficiënten voor E_zy  (afhankelijk van sigma_y, eps_r) ─
sy   = sigma_PML('Ezy') # shape (Nx, Ny)

denom_ezy = 1 + sy*dt/(2*eps0*epsr)
Ceya = (1 - sy*dt/(2*eps0*epsr)) / denom_ezy
Ceyb = (dt/(eps0*epsr)) / denom_ezy / dy     # inclusief 1/dy

print('Update-coëfficiënten berekend.')
print(f'Shapes: Chxa={Chxa.shape}, Chya={Chya.shape}')
print(f'Shapes: Cexa={Cexa.shape}, Ceya={Ceya.shape}')

Update-coëfficiënten berekend.
Shapes: Chxa=(2125, 2124), Chya=(2124, 2125)
Shapes: Cexa=(2125, 2125), Ceya=(2125, 2125)


In [219]:
# ── Veldmatrices initialiseren ─────────────────────────────────
Hx  = np.zeros((Nx+1, Ny))
Hy  = np.zeros((Nx, Ny+1))
Ezx = np.zeros((Nx+1, Ny+1))   # gesplitste E_z component (x-richting)
Ezy = np.zeros((Nx+1, Ny+1))   # gesplitste E_z component (y-richting)

# Observatie-tijdreeksen
Ez_obs    = np.zeros(Nt)
Je_t      = np.zeros(Nt)
Aantal_snap_times = 100
snap_times = np.linspace(0, Nt-1, Aantal_snap_times, dtype=int)   # tijdstappen om snapshot op te slaan
snapshots  = {}

print('Start tijdintegratie...')

for n in range(Nt):
    t_now = n * dt

    # ── Stap 1: Update H-velden ────────────────────────────────
    # H_x: i=0..Nx-1, j=0..Ny-2  (centrale diff in y-richting)
    Hx[:, :] = (Chxa[:, :] * Hx[:, :]
                  - Chxb[:, :] * (Ezx[:, 1:] + Ezy[:, 1:]
                                  - Ezx[:, :-1] - Ezy[:, :-1]))

    # H_y: i=0..Nx-2, j=0..Ny-1  (centrale diff in x-richting)
    Hy[:, :] = (Chya[:, :] * Hy[:, :]
                  + Chyb[:, :] * (Ezx[1:, :] + Ezy[1:, :]
                                   - Ezx[:-1, :] - Ezy[:-1, :]))

    # ── Stap 2: PEC-randvoorwaarden H ─────────────────────────
    # (Voor PEC-backed PML: tangentiële E=0 aan rand, geen extra H-rand nodig;
    #  de PML dempt de golven voor ze de PEC-rand bereiken)

    # ── Stap 3: Update E-velden (gesplitst) ────────────────────
    # E_zx: afhankelijk van dH_y/dx
    Ezx[1:-1, :] = (Cexa[1:-1, :] * Ezx[1:-1, :]
                  + Cexb[1:-1, :] * (Hy[1:, :] - Hy[:-1, :]))

    # E_zy: afhankelijk van -dH_x/dy en bron
    Ezy[:, 1:-1] = (Ceya[:, 1:-1] * Ezy[:, 1:-1]
                  - Ceyb[:, 1:-1] * (Hx[:, 1:] - Hx[:, :-1]))

    # ── Stap 4: Bron injecteren ────────────────────────────────
    # Gaussische puls: J_e(t) = J0 * exp(-(t-tc)^2 / (2*sigma_t^2))
    Je_t[n] = ps(t_now)
    # Bronterm: -J_e * dt/(eps0*eps_r) toegevoegd aan E_zy op bronlocatie
    Ezy[ps_x, ps_y] -= (dt / (eps0 * eps_r[ps_x, ps_y])) * ps(t_now)
    # (factor dy omdat Cezyb al 1/dy bevat; bronterm is J/(eps) * dt)

    # ── Stap 5: Totaal E_z ─────────────────────────────────────
    Ez = Ezx + Ezy

    # ── Stap 6: PEC-randvoorwaarden E ─────────────────────────
    # Tangentiële E_z = 0 op alle randen (PEC-backed PML)
    Ez[0, :]  = 0
    Ez[-1, :] = 0
    Ez[:, 0]  = 0
    Ez[:, -1] = 0
    # Synchroniseer Ezx/Ezy met de PEC-randvoorwaarden
    Ezx[0, :] = 0
    Ezy[0, :]  = -Ezx[0, :]
    Ezx[-1,:] = 0
    Ezy[-1, :]  = -Ezx[-1,:]
    Ezx[:, 0] = 0
    Ezy[:, 0]  = -Ezx[:, 0]
    Ezx[:,-1] = 0
    Ezy[:, -1]  = -Ezx[:, -1]

    # ── Observatie ────────────────────────────────────────────
    Ez_obs[n] = Ez[po_x, po_y]

    # Snapshots
    if n in snap_times:
        snapshots[n] = Ez.copy()

print('Tijdintegratie voltooid!')

Start tijdintegratie...
Tijdintegratie voltooid!


In [222]:
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle

time_axis = np.arange(Nt) * dt * 1e12  # in ps

fig_ts, axes_ts = plt.subplots(2, 1, figsize=(10, 5))
axes_ts[0].plot(time_axis, Je_t, 'C0')
axes_ts[0].set_xlabel('t [ps]')
axes_ts[0].set_ylabel('$J_e(t)$ [A/m²]')
axes_ts[0].set_title('Bronstroom (Gaussische puls)')
axes_ts[0].grid(True, alpha=0.3)

axes_ts[1].plot(time_axis, Ez_obs, 'C1')
axes_ts[1].set_xlabel('t [ps]')
axes_ts[1].set_ylabel('$E_z$ [V/m]')
axes_ts[1].set_title(f'Geobserveerd $E_z$ in punt ({po_x},{po_y}),lambda/60,7 min')
axes_ts[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

snap_list = sorted(snapshots.keys())

def get_vmax(snaps, keys, perc=99):
    all_vals = np.concatenate([snaps[n].ravel() for n in keys])
    vmax = np.percentile(np.abs(all_vals), perc)
    return vmax if vmax != 0 else 1

# Binnenste domein indices (zonder PML-cellen)
ix0, ix1 = N_PML, Nx - N_PML
iy0, iy1 = N_PML, Ny - N_PML

# Aparte snapshots voor binnenste domein
snapshots_inner = {n: snapshots[n][ix0:ix1, iy0:iy1] for n in snap_list}

vmax_full  = get_vmax(snapshots,       snap_list)  # volledige schaal
vmax_inner = get_vmax(snapshots_inner, snap_list)/20  # binnenste schaal (kleiner)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Links: volledig grid met PML ──────────────────────────────
Ez0_full = snapshots[snap_list[0]]
im_full = axes[0].imshow(Ez0_full.T, origin='lower', cmap='RdBu_r',
                          vmin=-vmax_full, vmax=vmax_full,
                          extent=[0, Nx*dx*1e3, 0, Ny*dy*1e3])
plt.colorbar(im_full, ax=axes[0], label='$E_z$ [V/m]')

# PML-grenzen
for lim in [N_PML*dx*1e3, (Nx-N_PML)*dx*1e3]:
    axes[0].axvline(lim, color='gray', lw=0.8, ls='--', alpha=0.6)
for lim in [N_PML*dy*1e3, (Ny-N_PML)*dy*1e3]:
    axes[0].axhline(lim, color='gray', lw=0.8, ls='--', alpha=0.6)

axes[0].plot(ps_x*dx*1e3, ps_y*dy*1e3, 'r*', ms=8, label='Bron')
axes[0].plot(po_x*dx*1e3, po_y*dy*1e3, 'bx', ms=8, label='Observatie')
if USE_OBJECT:
    rect0 = Rectangle((obj_x0*dx*1e3, obj_y0*dy*1e3),
                       (obj_x1-obj_x0)*dx*1e3, (obj_y1-obj_y0)*dy*1e3,
                       fill=False, edgecolor='green', lw=1.5, label='Object')
    axes[0].add_patch(rect0)
axes[0].legend(loc='upper right', fontsize=8)
axes[0].set_xlabel('x [mm]')
axes[0].set_ylabel('y [mm]')
title_full = axes[0].set_title('')

# ── Rechts: binnenste domein zonder PML-cellen ───────────────
Ez0_inner = snapshots_inner[snap_list[0]]
im_inner = axes[1].imshow(Ez0_inner.T, origin='lower', cmap='RdBu_r',
                           vmin=-vmax_inner, vmax=vmax_inner,
                           extent=[ix0*dx*1e3, ix1*dx*1e3,
                                   iy0*dy*1e3, iy1*dy*1e3])
plt.colorbar(im_inner, ax=axes[1], label='$E_z$ [V/m]')

axes[1].plot(ps_x*dx*1e3, ps_y*dy*1e3, 'r*', ms=8, label='Bron')
axes[1].plot(po_x*dx*1e3, po_y*dy*1e3, 'bx', ms=8, label='Observatie')
if USE_OBJECT:
    rect1 = Rectangle((obj_x0*dx*1e3, obj_y0*dy*1e3),
                       (obj_x1-obj_x0)*dx*1e3, (obj_y1-obj_y0)*dy*1e3,
                       fill=False, edgecolor='green', lw=1.5, label='Object')
    axes[1].add_patch(rect1)
axes[1].legend(loc='upper right', fontsize=8)
axes[1].set_xlabel('x [mm]')
axes[1].set_ylabel('y [mm]')
title_inner = axes[1].set_title('')

def update(frame):
    n = snap_list[frame]
    t_ps = n * dt * 1e12
    im_full.set_data(snapshots[n].T)
    im_inner.set_data(snapshots_inner[n].T)
    title_full.set_text(f'Volledig grid (met PML) — t = {t_ps:.1f} ps')
    title_inner.set_text(f'Binnenste domein — t = {t_ps:.1f} ps')
    return [im_full, im_inner, title_full, title_inner]

anim = FuncAnimation(fig, update, frames=len(snap_list),
                     interval=150, blit=False, repeat=True)

plt.suptitle('$E_z$ [V/m] — links: volledig grid, rechts: zonder PML-zone')
plt.tight_layout()
plt.show()

# anim.save('pml_vergelijking.gif', writer='pillow', fps=8)
# anim.save('pml_vergelijking.mp4', writer='ffmpeg', fps=10, dpi=150)

In [221]:
# # ── FFT van geobserveerd veld en bron ─────────────────────────
# Nfft   = 2*Nt    # zero-padding voor betere frequentieresolutie
# freqs  = np.fft.rfftfreq(Nfft, d=dt)   # [Hz]
# omegas = 2*np.pi*freqs

# Ez_fft = np.fft.rfft(Ez_obs, n=Nfft)
# Je_fft = np.fft.rfft(Je_t,   n=Nfft)

# # Bronspectrum (analytisch voor Gaussische puls)
# Je_spectrum_analytic = (J0 * sigma * np.sqrt(2*np.pi)
#                         * np.exp(-sigma**2 * omegas**2 / 2)
#                         * np.exp(-1j * omegas * tc))

# # Frequentierespons = E_z(omega) / J_e(omega)
# # Beperk tot bandbreedte van bron (omega < 3/sigma_t)
# omega_max  = 5 / sigma
# f_max      = omega_max / (2*np.pi)
# band_mask  = (freqs > 0) & (freqs <= f_max)

# # Transferfunctie (FDTD)
# H_fdtd = np.zeros_like(Ez_fft, dtype=complex)
# H_fdtd[band_mask] = Ez_fft[band_mask] / Je_spectrum_analytic[band_mask]

# # ── Analytische Hankel-functie ─────────────────────────────────
# xs_m = ps_x * dx
# ys_m = ps_y * dy
# xo_m = po_x * dx
# yo_m = po_y * dy
# r    = np.sqrt((xo_m - xs_m)**2 + (yo_m - ys_m)**2)

# f_plot = freqs[band_mask]
# om_plot = 2*np.pi*f_plot
# k0_plot = om_plot / c

# Ez_hankel = -J0 * om_plot * mu0 / 4 * hankel2(0, k0_plot * r)

# # ── Plot vergelijking ──────────────────────────────────────────
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# axes[0].plot(f_plot/1e9, np.abs(H_fdtd[band_mask]),
#              label='Yee-FDTD', lw=2)
# axes[0].plot(f_plot/1e9, np.abs(Ez_hankel),
#              '--', label='Hankel (analytisch)', lw=2)
# axes[0].set_xlabel('f [GHz]')
# axes[0].set_ylabel('$|E_z(f)|$ [V·m⁻¹·(A·m⁻²)⁻¹·Hz]')
# axes[0].set_title('Amplitude-spectrum — validatie')
# axes[0].legend()
# axes[0].grid(True, alpha=0.3)

# axes[1].plot(f_plot/1e9, np.angle(H_fdtd[band_mask]),
#              label='Yee-FDTD', lw=2)
# axes[1].plot(f_plot/1e9/2/np.pi, np.angle(Ez_hankel),
#              '--', label='Hankel (analytisch)', lw=2)
# axes[1].set_xlabel('f [GHz]')
# axes[1].set_ylabel('Fase [rad]')
# axes[1].set_title('Fase-spectrum — validatie')
# axes[1].legend()
# axes[1].grid(True, alpha=0.3)

# plt.suptitle(f'Observatiepunt op r = {r*1e3:.1f} mm van de bron'
#              + (' (ZONDER object)' if not USE_OBJECT else ' (MET object — validatie enkel geldig in vrije ruimte)'))
# plt.tight_layout()
# plt.show()

# # Relatieve fout
# rel_err = (np.abs(H_fdtd[band_mask] - Ez_hankel)
#            / (np.abs(Ez_hankel) + 1e-30))
# print(f'Gemiddelde relatieve fout (amplitude): {rel_err.mean()*100:.2f}%')